# Summary of lecture 3

Yesterday we discretized a random dynamical system with additive uniform noise on the Ulam basis, we bounded the norms of the powers of the discretized annealed operator on the space of vectors of average zero, we transported those bounds from a coarse partition to a fine one, and we turned the residual of an approximate fixed point into a rigorous $L^1$ bound on the distance from the stationary density.
With that bound and a splitting of the space we enclosed the Lyapunov exponent of a family of unimodal maps at two noise sizes, one enclosure positive and the other negative.

Those norms are a mixing rate: a certified statement that after $n$ steps the operator has contracted every zero average density by a factor $\eta_n$.
Today the mixing rate is the object we go after, and the density and the Lyapunov exponent come out of it.

# A certified mixing rate

We work with the family
$$
T_{\alpha,\beta}(x) = \beta - (1+\beta)|x|^{\alpha}, \qquad \alpha \geq 1,\ \beta \in (-1, 1],
$$
on $[-1,1]$, perturbed by additive Gaussian noise of standard deviation $\sigma$, with the boundary condition that identifies $x$ with $x+2$; the technique is the one of [Galatolo, Lopez Vereau, Marangio, Nisoli, *Efficient computation of stationary measures and the Lyapunov landscape for families of random dynamical systems with smooth additive noise*](https://doi.org/10.1137/25M1786441), and the implementation is the one packaged in `PlateauExperiment.jl`.

The annealed transfer operator is again $P_\sigma f = \rho_\sigma * (Pf)$, and the reason to change basis is that convolution is multiplication frequency by frequency:
$$
\mathcal{F}(\rho_{\sigma}*f)[k] = e^{-\sigma^2 k^2 \pi^2/2}\,\mathcal{F}(f)[k].
$$
Since $P$ is a weak contraction of $L^1$, every $f \in L^1$ is sent by $P_\sigma$ to a function whose Fourier coefficients decay like $e^{-\sigma^2 k^2\pi^2/2}$; the operator is smoothing, the truncation at frequency $K$ costs an error that is exponentially small in $K^2$, and a matrix of size $2K+1$ is enough.

## Setting up

In [ ]:
import Pkg
Pkg.activate("./")
Pkg.add(["RigorousInvariantMeasures", "BallArithmetic", "IntervalArithmetic",
         "FFTW", "TaylorModels", "Plots", "RecipesBase", "LaTeXStrings"])

In [ ]:
using RigorousInvariantMeasures, IntervalArithmetic, BallArithmetic
using LinearAlgebra, FFTW, TaylorModels
using Plots, RecipesBase, LaTeXStrings

import IntervalArithmetic: inf, sup, mid, radius, diam, interval

setdisplay(:infsup; decorations = false, ng_flag = false)

We fix the parameters and build the basis. `FourierAdjoint(K, FFTNx)` is a Fourier basis truncated at the frequencies $-K, \dots, K$, which assembles the transfer operator through a discretization of its adjoint; the adjoint is the composition operator $f \mapsto f\circ T$, so the dynamic may be handed over as an ordinary function rather than as a piecewise map with its branches. The change of variables sends the system from $[-1,1]$ to $[0,1]$.

In [ ]:
α = interval(3.5)
β = interval(1)
K = 128
FFTNx = 1024

B = FourierAdjoint(K, FFTNx)

In [ ]:
τ₁(x) = (x+1)/2          # from [-1, 1] to [0, 1]
τ₂(x) = 2*x-1            # from [0, 1] to [-1, 1]

T_pm(x) = β - (1+β)*abs(x)^α
T(x) = τ₁(T_pm(τ₂(x)))

In [ ]:
plot(x -> IntervalArithmetic.mid(T(x)), 0, 1; label = "T on [0,1]")

In [ ]:
@time PK = assemble(B, T)

## Ball arithmetic

The assembled matrix is a matrix of complex intervals, and every operation we are about to perform on it is linear algebra: products, an eigendecomposition, singular values.
An interval matrix carries two endpoints per real and per imaginary part and cannot use the machine's matrix multiplication; a ball matrix carries a centre matrix and a radius matrix, so the centre product is one call to the tuned floating point routine and the radius is bounded afterwards by a few norm inequalities. We convert once, and we stay in ball arithmetic for the rest of the notebook; loading `IntervalArithmetic` beside `BallArithmetic` brings in the conversion, so `BallMatrix` applied to a matrix of intervals is all that we need, and on a complex matrix it returns the disc that contains the rectangle, with the radius rounded up.

In [ ]:
bPK = BallMatrix(PK)
maximum(bPK.r)

Convolution with the noise is diagonal in this basis, with the entries written above; we build it as an interval matrix and convert it the same way.

In [ ]:
σ = interval(0.1)

NoiseInterval(σ, K) = Diagonal([[exp((-σ^2*interval(π)^2*interval(k)^2)/2) for k in 0:K];
                                [exp((-σ^2*interval(π)^2*interval(k)^2)/2) for k in -K:-1]])
bD = BallMatrix(NoiseInterval(σ, K))

In [ ]:
PσK = bD*bPK

The two heatmaps below plot $-\log|M_{ij}|$, so a light colour means an entry that is exponentially small; the second one shows what the noise does to the high frequencies. The colour scales are not the same.

In [ ]:
heatmap(-log.(abs.(BallArithmetic.mid.(bPK))); title = "deterministic")

In [ ]:
heatmap(-log.(abs.(BallArithmetic.mid.(PσK))); title = "with noise")

## The stationary density

We compute an approximate fixed point from the centre matrix, with an ordinary eigendecomposition; nothing here needs to be rigorous, since the error will be measured afterwards by the residual.

In [ ]:
F = eigen(PσK.c)
p = sortperm(abs.(F.values), rev = true)
F.values[p][1:5]

In [ ]:
scatter(F.values; label = "eigenvalues of the centre matrix")
plot!([cos(t) for t in 0:0.01:2π], [sin(t) for t in 0:0.01:2π]; label = "unit circle")

The leading eigenvalue is $1$, since the operator preserves the integral, and we normalise the corresponding eigenvector so that its zeroth coefficient is $1$.
The Fourier coefficients of a real function satisfy $\hat f[-k] = \overline{\hat f[k]}$, and the numerical eigenvector loses that symmetry; we impose it, which costs nothing and makes the density real.

In [ ]:
fσK = F.vectors[:, p[1]]
fσK /= fσK[1]

function symmetrise(v)
    w = zeros(eltype(v), length(v))
    N = (length(v)-1) ÷ 2
    w[1:N+1] = v[1:N+1]
    w[end-N+1:end] = [x' for x in reverse(v[2:N+1])]
    return w
end

fσKs = symmetrise(fσK)
bf = BallVector(fσKs)

In [ ]:
plot(real.(ifft(BallArithmetic.mid(bf))); label = "stationary density")

The residual is computed in ball arithmetic, and it is the only place where the approximate eigenvector meets a rigorous operation; $\epsilon$ below bounds $\|P_{\sigma,K} f_{\sigma,K,s} - f_{\sigma,K,s}\|_{L^2}$.

In [ ]:
res = PσK*bf - bf
ε = norm(res.c, 2) + norm(res.r, 2)

# The mixing rate

Let $V$ be the subspace of $L^2$ of average zero, which is invariant under the annealed operator; a mixing rate is a pair $(n, \eta)$ with
$$
\big\|P_{\sigma,K}^{\,n}\big|_V\big\|_{L^2\to L^2} \leq \eta < 1,
$$
proved, not observed. In the truncated basis the restriction to $V$ is the matrix with the first row and the first column removed, since the zeroth coefficient is the integral.

The eigenvalues we plotted give a candidate answer, the second largest modulus; that answer is wrong at finite $n$, and we shall see by how much.

In [ ]:
A = PσK[2:end, 2:end]
size(A)

`svd_bound_L2_opnorm` returns a rigorous upper bound for the largest singular value of a ball matrix, computed from a floating point singular value decomposition and a perturbation bound; we apply it to the powers.

In [ ]:
function power_norms(A, N)
    norms = zeros(N)
    Ai = A
    for i in 1:N
        norms[i] = BallArithmetic.svd_bound_L2_opnorm(Ai)
        Ai = Ai*A
    end
    return norms
end

In [ ]:
@time norms = power_norms(A, 12)

In [ ]:
plot(1:12, norms; yscale = :log10, marker = :circle,
     label = "certified bound for the norm of the n-th power")

In [ ]:
n₁ = findfirst(<(1.0), norms)
η  = interval(norms[n₁])
n₁, η

The first power is expansive, with norm above $1$; the second contracts, and $(n_1, \eta) = (2, 0.9193)$ is the first certified mixing rate.
Taking the twelfth power instead gives a much better rate per step, and the comparison with the second eigenvalue is the following.

In [ ]:
norms[end]^(1/12), maximum(abs.(F.values[p][2:end]))

## Why the spectrum is not the answer

The operator is not normal, so its spectrum does not control the norms of its powers at finite time; the set of points where the resolvent exceeds $1/\varepsilon$, drawn below for a coarser truncation, reaches outside the unit circle although every eigenvalue lies well inside it, which is why $\|A\|$ is above one while the spectral radius is about one half.

In [ ]:
Ksmall = 40
Bs = FourierAdjoint(Ksmall, 1024)
PKs = assemble(Bs, T)
As = (BallMatrix(NoiseInterval(σ, Ksmall))*BallMatrix(PKs)).c[2:end, 2:end]
size(As)

In [ ]:
xs = range(-1.3, 1.3; length = 90)
ys = range(-1.3, 1.3; length = 90)
@time Z = [minimum(svdvals(As - (x+im*y)*I)) for y in ys, x in xs]
contour(xs, ys, log10.(Z); levels = -6:0.25:0.5, aspect_ratio = 1,
        title = "log10 of the smallest singular value of zI - A")
plot!([cos(t) for t in 0:0.01:2π], [sin(t) for t in 0:0.01:2π]; label = "unit circle")

The figure is drawn from floating point singular values and is not certified; the numbers above it are.

# From the mixing rate to the density

We use the following statement. Let $f_\sigma$ be the fixed point of $P_\sigma$, let $f_{\sigma,K,s}$ be the symmetrized approximate fixed point computed above, let $\epsilon$ bound its residual, and suppose there are $n$ and $\eta < 1$ with $\|P_{\sigma,K}^n|_V\|_{L^2\to L^2}\leq \eta$ and constants $C_i \geq \max(1, \|P_{\sigma,K}^i|_V\|)$ for $0 \leq i \leq n-1$. Then
$$
\|f_{\sigma}-f_{\sigma,K,s}\|_{L^{2}} \leq \frac{1}{1-\eta}\sum_{i=0}^{n-1}C_i\big(\delta_{\sigma,K} + \epsilon\big),
$$
where $\delta_{\sigma,K}$ collects the error made by truncating the Fourier expansion at $K$, namely
$$
\delta_{\sigma,K} = \Gamma_{\sigma,K}(1+\Gamma^1_{\sigma,K}) + \|\rho_\sigma\|_{L^2}\sqrt{\coth(1/2\sigma^2)}\,\Gamma^1_{\sigma,K},
$$
with $\Gamma_{\sigma,K}$ the $L^1 \to L^2$ tail bound and $\Gamma^1_{\sigma,K}$ the $L^1\to L^1$ one; both carry the factor $e^{-\sigma^2K^2\pi^2/2}$.

Both $n$ and the $C_i$ come out of the computation just done, and are not free parameters: $n$ is the first index at which the certified norm drops below one, $\eta$ is the norm there, and the $C_i$ are the earlier norms, raised to $1$ where they are smaller.

In [ ]:
C = [interval(1.0); [interval(max(1.0, norms[i])) for i in 1:n₁-1]]

In [ ]:
Iπ = interval(π)

Γ  = sqrt(coth(1/(2*σ^2))/(σ*sqrt(Iπ)))*exp((-σ^2*interval(K)^2*Iπ^2)/2)
Γ¹ = (2/(σ^2*Iπ^2*interval(K)))*exp((-σ^2*interval(K)^2*Iπ^2)/2)
ρ₂ = sqrt(1/(2*σ*sqrt(Iπ)))*sqrt(coth(1/(2*σ^2)))

δ = Γ*(1+Γ¹) + ρ₂*Γ¹

At $\sigma = 0.1$ and $K = 128$ the exponent $-\sigma^2K^2\pi^2/2$ is about $-808$, so $\Gamma$ underflows the smallest positive double and the enclosure of $\delta$ is $[0, 3\cdot 10^{-323}]$; the truncation contributes nothing here, and the residual $\epsilon$ is what the bound is made of.

In [ ]:
R = (sum(C)*(δ+interval(ε)))/(1-η)

# The Lyapunov exponent

The Lyapunov exponent of the random system is
$$
\lambda(\alpha,\beta,\sigma) = \int_{-1}^{1}\log|T'_{\alpha,\beta}|\,f_{\sigma}\,dm,
$$
and, since we work with Fourier coefficients, we need those of the observable:
$$
\mathcal{F}(\log|T'|)[0] = \log((1+\beta)\alpha)-(\alpha-1),\qquad
\mathcal{F}(\log|T'|)[j] = -\frac{\alpha-1}{j\pi}\int_0^{j\pi}\frac{\sin t}{t}\,dt.
$$
We enclose the integral by splitting it at the zeros of the sine: on $[0,\pi]$ we sum the alternating power series of the primitive of $\sin(t)/t$ and bound the remainder by the first omitted term, and on each later arch $[i\pi, (i+1)\pi]$ we use a Taylor model, exactly as in lecture 2.
We work in $256$ bits, since the power series at $0$ has terms with factorials in the denominator.

In [ ]:
setprecision(256)
Pi = interval(BigFloat, π)

sinc_over(t) = sin(t)/t
tay(i, x) = x^(2i+1)/(interval(BigFloat, factorial(big(2i+1)))*interval(BigFloat, 2i+1))

In [ ]:
Nser = 60
I₀ = sum([(-1)^i*tay(i, Pi) for i in 0:Nser]) + interval(BigFloat, -1, 1)*abs(tay(Nser+1, Pi))

In [ ]:
function int_over_arch(f, i; degree = 40)
    I = interval(inf(interval(BigFloat, i)*Pi), sup(interval(BigFloat, i+1)*Pi))
    m = interval(BigFloat, mid(I))
    prim = TaylorSeries.integrate(f(TaylorModel1(degree, m, I)))
    return prim(sup(I)-m) - prim(inf(I)-m)
end

In [ ]:
@time arcs = [I₀; [int_over_arch(sinc_over, i) for i in 1:K-1]]
maximum(diam.(arcs))

The cumulative sum of the arches gives $\int_0^{j\pi}\sin(t)/t\,dt$; dividing by $j\pi$ gives the coefficients. The alternating sign that follows converts the coefficients on $[-1,1]$ into the coefficients on $[0,1]$, which is the coordinate change we made at the beginning.

In [ ]:
coeff = cumsum(arcs) ./ [interval(BigFloat, i)*Pi for i in 1:K]
coeff01 = [(-1)^i for i in 1:K] .* coeff

αb = interval(BigFloat, 3.5)
βb = interval(BigFloat, 1)
lnn = [log((1+βb)*αb) - (αb-1); -(αb-1)*[coeff01; reverse(coeff01)]]
length(lnn), lnn[1], lnn[2]

In [ ]:
λK = sum(lnn[i]*(interval(BigFloat, real(fσKs[i])) + im*interval(BigFloat, imag(fσKs[i])))
         for i in 1:2K+1)
real(λK)

The last ingredient turns the $L^2$ error on the density into an error on the integral: by Cauchy-Schwarz,
$$
|\lambda(\alpha,\beta,\sigma) - \langle \log|T'|, f_{\sigma,K,s}\rangle| \leq \Upsilon\,\|f_\sigma - f_{\sigma,K,s}\|_{L^2},
\qquad \Upsilon = \|\log|T'|\|_{L^2},
$$
and for this family $\Upsilon$ is known in closed form,
$$
\Upsilon(\alpha,\beta) = \sqrt{2}\Big(\big(\log((1+\beta)\alpha)-(\alpha-1)\big)^2+(\alpha-1)^2\Big)^{1/2}.
$$

In [ ]:
Υ = sqrt(interval(BigFloat, 2))*((log((βb+1)*αb)-(αb-1))^2 + (αb-1)^2)^interval(BigFloat, 0.5)

In [ ]:
Rb = interval(BigFloat, inf(R), sup(R))
λ = real(λK) + Υ*Rb*interval(BigFloat, -1, 1)

In [ ]:
diam(λ)

The enclosure is about $2\cdot 10^{-10}$ wide, and it is positive: at $\alpha = 3.5$, $\beta = 1$ and $\sigma = 0.1$ the orbits of this random system separate.
Every link in the chain came from a certified number: the matrix from interval arithmetic, the mixing rate from certified singular values, the residual and the density error from ball arithmetic, the observable from Taylor models.

# The peripheral spectrum of a map with a stable cycle

The logistic map $T(x) = rx(1-x)$ at $r = 3.83$ lies inside the period three window, which opens at $1+\sqrt{8} = 3.8284\ldots$; the orbit
$$
0.156149 \mapsto 0.504666 \mapsto 0.957417 \mapsto 0.156149
$$
is attracting, its multiplier $\prod_i T'(x_i)$ being $0.32988$.
Additive noise turns the attracting cycle into a stationary density with three peaks, and the random orbit goes round the three peaks in order for a long time before it loses count; the eigenvalues of the annealed operator that sit near the unit circle are what measures that time.
We keep the periodic identification of $0$ with $1$, so that the third point of the cycle, at $0.9574$, lies within a few noise widths of the identification and part of the mass that leaves the cycle wraps round to the other side.

We compute the peripheral eigenvalues twice: first those of the matrix, to nine digits, and then those of the operator, to four.
The two are different objects, and the second is the one the dynamics is about.

In [ ]:
Klog = 128
NX = 2^18

rlog = interval(3.83)
Tlog(x) = rlog*x*(1-x)

plot(x -> IntervalArithmetic.mid(Tlog(x)), 0, 1; label = "T at r = 3.83")
plot!(x -> IntervalArithmetic.mid(Tlog(Tlog(Tlog(x)))), 0, 1; label = "its third iterate")
plot!(x -> x, 0, 1; label = "", color = :black, linestyle = :dash)

The three points where the third iterate crosses the diagonal with a small slope are the cycle.

`FourierAdjoint(K, N)` samples the integrand of
$$
P_{k,\ell} = \frac{1}{2}\int_{-1}^{1} e_k(T(x))\,e_\ell(x)\,dx, \qquad e_k(x) = e^{k\pi i x},
$$
on $N$ equispaced points and takes the discrete transform, so the entries it returns are exact up to the aliasing
$$
\mathcal{F}_N(f)[\ell] - \mathcal{F}(f)[\ell] = \sum_{q\neq 0}\mathcal{F}(f)[\ell + qN],
$$
which the interval radii of the assembled matrix do not cover: those radii are $4\cdot 10^{-13}$ and track the arithmetic alone.
The map is continuous at the identification and its derivative is not, $T'$ jumping by $2r$ there, so $f_k = e_k\circ T$ has a corner and its coefficients decay like $|\ell|^{-2}$ rather than exponentially; the trapezoidal rule is second order and the oversampling has to be generous.
We take $N = 2^{18}$ for $K = 128$, and we bound the aliasing below, since it is the quantity that decides how sharply the operator eigenvalues can be enclosed.

In [ ]:
Blog = FourierAdjoint(Klog, NX)
@time PKlog = assemble(Blog, Tlog)
bPKlog = BallMatrix(PKlog)
maximum(bPKlog.r)

We compute the eigenvalues of the matrix numerically and treat them as an oracle: they say where to draw the circles and they enter nothing else. The certificate below takes a centre and a radius, bounds $\sigma_{\min}(zI-\widetilde A)$ on the circle and carries that bound to the operator; the centre may be any complex number, so an oracle that is merely plausible does the job, and no verified eigenvalue algorithm is needed.

In [ ]:
function peripheral(bP, σ, K)
    A = BallMatrix(NoiseInterval(σ, K))*bP
    F = eigen(BallArithmetic.mid(A))
    p = sortperm(abs.(F.values), rev = true)
    return (A = A, F = F, p = p)
end

In [ ]:
σs = [0.02, 0.03, 0.05, 0.08, 0.12]
@time results = [peripheral(bPKlog, interval(s), Klog) for s in σs]
length(results)

The plot below shows the computed eigenvalues; the crosses are the three cube roots of $1$.

In [ ]:
pl = plot([cos(t) for t in 0:0.01:2π], [sin(t) for t in 0:0.01:2π];
          label = "", color = :black, aspect_ratio = 1, legend = :outerright)
scatter!(pl, [1.0, cos(2π/3), cos(4π/3)], [0.0, sin(2π/3), sin(4π/3)];
         label = "cube roots of 1", marker = (:xcross, 7), color = :black)
for (s, res) in zip(σs, results)
    λs = res.F.values[res.p[1:4]]
    scatter!(pl, real.(λs), imag.(λs); label = "σ = $s", marker = 5)
end
pl

The three peaks of the stationary density at the smallest noise size are the three points of the cycle.

In [ ]:
res₀ = results[1]
f₀ = symmetrise(res₀.F.vectors[:, res₀.p[1]]/res₀.F.vectors[1, res₀.p[1]])
plot(range(0, 1; length = 2Klog+2)[1:end-1], real.(ifft(f₀));
     label = "stationary density at σ = 0.02")

# From the matrix to the operator

Nothing so far is a statement about $P_\sigma$: an eigenvalue of a $257\times 257$ matrix is an eigenvalue of a $257\times 257$ matrix.

The bridge to the operator is one inequality. Write $P_{\sigma,k}$ for the finite rank operator we actually hold, $R(z, P_{\sigma,k}) = \|(z-P_{\sigma,k})^{-1}\|$ for its resolvent, and let $\delta_k$ be a number with $\delta_k \geq \|P_\sigma - P_{\sigma,k}\|$. If
$$
R(z, P_{\sigma,k})\,\delta_k < 1 \qquad \text{for every } z \text{ on a closed contour } \Gamma,
$$
then $\Gamma$ lies in the resolvent set of $P_\sigma$, and $P_\sigma$ has inside $\Gamma$ exactly as many eigenvalues, counted with multiplicity, as $P_{\sigma,k}$ has. The reason is that the whole segment $P_{\sigma,k} + t(P_\sigma - P_{\sigma,k})$, $t\in[0,1]$, keeps $\Gamma$ in its resolvent set under the same inequality, so its Riesz projector is continuous in $t$, and a norm continuous family of projectors has constant rank.

Everything else is the work of producing the two numbers. Here $\delta_k$ splits into the Fourier tail $\tau_K$, exponentially small in $\sigma^2K^2$, and the aliasing $\varepsilon_{\rm alias}$ of the assembler, which falls like $N^{-2}$; a third constant $b_K$ bounds the coupling between the truncated modes and the rest, and $R$ comes from the certified singular values below. We compute the three constants and take their estimates from [Nisoli](https://arxiv.org/abs/2602.19435), [Blumenthal, Nisoli, Taylor-Crush](https://arxiv.org/abs/2507.09021) and [Galatolo, Lopez Vereau, Marangio, Nisoli](https://doi.org/10.1137/25M1786441), where they are derived.

In [ ]:
# The three constants of the bridge. Ck is the C^2 constant of e_k∘T on [-1,1] for
# T(y) = (r/2)(1-y^2)-1, whose derivative jumps by 2r at the identification; the
# aliasing of the N-point transform is then π²Ck/(3N²) per entry, damped by the noise
# row by row, and we take the Frobenius norm of that to bound the 2-norm.
Ck(k, r) = 2*abs(interval(k))*r/interval(π) + interval(k)^2*r^2/interval(3)

alias_bound(K, N, σ, r) = sqrt(interval(2K+1)*sum(
    exp(-σ^2*interval(π)^2*interval(k)^2)*(interval(π)^2*Ck(k, r)/(interval(3)*interval(N)^2))^2
    for k in -K:K))

# τ_K bounds the modes above K, using |F(P f)[k]| ≤ exp(-σ²π²k²/2)‖f‖ and
# k² ≥ (K+1)² + (2K+1)(k-K-1) to sum the tail.
function tail_bound(K, σ)
    c = σ^2*interval(π)^2
    return sqrt(interval(2)*exp(-c*interval(K+1)^2)/(interval(1) - exp(-c*interval(2K+1))))
end

# b_K bounds the coupling block of the truncation, which is block triangular.
coupling_bound(K, σ) = sqrt(sum(exp(-σ^2*interval(π)^2*interval(k)^2) for k in -K:K))

In [ ]:
σ₀ = interval(0.02)
εa = sup(alias_bound(Klog, NX, σ₀, rlog))
τK = sup(tail_bound(Klog, σ₀))
bK = sup(coupling_bound(Klog, σ₀))
(εa, τK, bK)

The contour is a circle of radius $\rho$ around a computed eigenvalue, covered by $n$ overlapping discs: the points of the circle nearest to $z_j = \lambda + \rho e^{2\pi i j/n}$ are within $2\rho\sin(\pi/2n)$ of it, so discs of that radius centred at the $n$ points cover the circle, and one certified singular value decomposition per disc bounds $\sigma_{\min}(zI-\widetilde A)$ from below over the whole disc, hence over the whole contour.
The decomposition is certified by `svdbox`, whose `RumpOriginal` and `MiyajimaM1` methods return the same bound here, the first at 0.091 seconds a disc against 0.107; both start from a floating point decomposition and bound the singular values of every matrix in the ball by a perturbation argument, so the radius of the disc is charged to $\sigma_{\min}$ once and for all.

The two parts of $\delta_k$ enter in different places. The aliasing perturbs the matrix alone, so it is absorbed there,
$$
\|(z-A)^{-1}\| \;\leq\; \frac{\widetilde R}{1 - \widetilde R\,\varepsilon_{\rm alias}},
\qquad \widetilde R = \frac{1}{\sigma_{\min}(zI-\widetilde A)},
$$
which asks $\sigma_{\min} > \varepsilon_{\rm alias}$ on the contour. The tail perturbs the operator, so it is absorbed at the end, and the inequality above takes the form
$$
\alpha := \tau_K \sup_{z\in\Gamma}\|(z-Q)^{-1}\| < 1 .
$$

In [ ]:
function necklace(A, λ, ρ, n; method = RumpOriginal())
    pearl = sup(interval(2)*interval(ρ)*sin(interval(π)/interval(2n)))
    σmin = Inf
    for j in 0:n-1
        s = svdbox(A - Ball(λ + ρ*cis(2π*j/n), pearl)*I; method = method, apply_vbd = false)[end]
        σmin = min(σmin, s.c - s.r)
    end
    return σmin, pearl
end

function bridge(σmin, εa, τK, bK, absz)
    R̃ = interval(1)/interval(σmin)
    gap = interval(1) - R̃*interval(εa)
    inf(gap) <= 0 && return (Inf, Inf)
    R = R̃/gap
    RQ = sqrt(R^2*(interval(1) + (interval(bK)/interval(absz))^2) + interval(1)/interval(absz)^2)
    return sup(RQ), sup(interval(τK)*RQ)
end

In [ ]:
function certify_on_circle(res, λ, ρ, σ; npearls = 128)
    εa = sup(alias_bound(Klog, NX, σ, rlog))
    τK = sup(tail_bound(Klog, σ))
    bK = sup(coupling_bound(Klog, σ))
    σmin, pearl = necklace(res.A, λ, ρ, npearls)
    RQ, α = bridge(σmin, εa, τK, bK, abs(λ) - ρ)
    println(round(λ, digits = 7), "   ρ = ", ρ,
            "   disc = ", round(pearl, sigdigits = 3),
            "   σmin ≥ ", round(σmin, sigdigits = 3),
            "   R̃ εₐ = ", round(εa/σmin, sigdigits = 3),
            "   ‖(z-Q)⁻¹‖ ≤ ", round(RQ, sigdigits = 4),
            "   α = ", round(α, sigdigits = 3),
            α < 1 ? "   proved" : "   fails")
end

@time begin
    r₀ = results[1]                                    # σ = 0.02
    certify_on_circle(r₀, r₀.F.values[r₀.p[1]], 3e-5, interval(0.02))
    certify_on_circle(r₀, r₀.F.values[r₀.p[2]], 1e-4, interval(0.02))
    r₁ = results[end]                                  # σ = 0.12
    certify_on_circle(r₁, r₁.F.values[r₁.p[2]], 1e-5, interval(0.12))
end

So the annealed transfer operator of the logistic map at $r = 3.83$ with Gaussian noise of size $\sigma = 0.02$ has exactly one eigenvalue within $3\cdot 10^{-5}$ of $1$, which it must, and exactly one within $10^{-4}$ of $-0.370874 + 0.542360i$; the conjugate statement holds at the conjugate point, the matrix being real in the sense that $A_{-j,-k} = \overline{A_{j,k}}$.
At $\sigma = 0.12$ the same argument puts one eigenvalue within $10^{-5}$ of $-0.424329$.
These are the only certified statements of the section, and they are statements about the operator; the centres come from the oracle and are floating point numbers.

The circle around $\lambda_2$ at $\sigma = 0.02$ cannot be shrunk to $3\cdot 10^{-5}$: there $\sigma_{\min}$ falls to $2.3\cdot 10^{-6}$, below $\varepsilon_{\rm alias} = 4.5\cdot 10^{-6}$, and the first step of the bridge fails.
What limits the enclosure is the oversampling of the assembler and nothing else: $\varepsilon_{\rm alias}$ falls like $N^{-2}$, so four times the sampling would divide it by sixteen, and the Fourier tail $\tau_K$, which is $10^{-14}$ here and $10^{-162}$ at $\sigma = 0.12$, is never the binding constraint.

The computation does not say that these are the eigenvalues of largest modulus of $P_\sigma$; that needs a contour separating them from the rest, and the same machinery would prove it at the price of a necklace on a large circle.

# Summary of the lecture

We discretized the annealed transfer operator of a random system with Gaussian noise on a Fourier basis, where the noise is diagonal and the truncation error is exponentially small in $K^2$, and we worked throughout in ball arithmetic.

From certified singular values of the powers of the operator restricted to the zero average subspace we obtained a mixing rate, a proved pair $(n, \eta)$ with $\|P^n|_V\| \leq \eta < 1$; the first power has norm above one although the spectral radius is about one half, which is what non-normality does and what the pseudospectrum picture shows.

The mixing rate, the residual of an approximate fixed point and the truncation constants gave an $L^2$ bound on the distance between the computed density and the true one, and the Fourier coefficients of $\log|T'|$, enclosed with Taylor models, turned that into an enclosure of the Lyapunov exponent.

The closing section took the logistic map at $r = 3.83$, a parameter inside the period three window, and enclosed the peripheral eigenvalues of its annealed operator, not of its matrix: a necklace of overlapping discs around a circle, one Rump certified singular value decomposition per disc, and a small gain condition that carries the bound from the truncation to the operator. What limits the enclosure is the aliasing of the Fourier assembler, bounded by the estimate of the paper above and falling like $N^{-2}$ in the oversampling.